# MLP Feature Movement Check

This notebook shows the underlying affordability features moving over time. The goal is to make the regime shift visible directly from the data, without relying on model outputs.

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'outputs').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_FILE = PROJECT_ROOT / 'outputs' / 'bayesian_data.csv'

bayes = pd.read_csv(DATA_FILE)
bayes['date'] = pd.to_datetime(bayes['date'])
bayes['month'] = bayes['date'].dt.to_period('M').dt.to_timestamp()

monthly = (
    bayes.groupby('month', as_index=False)
    .agg(
        income_needed=('income_needed', 'mean'),
        mortgage=('MORTGAGE30US', 'mean'),
        inventory=('inventory', 'mean'),
        days_to_pending=('days_to_pending', 'mean'),
        zhvi_yoy_pct=('zhvi_yoy_pct', 'mean'),
    )
    .sort_values('month')
)

threshold = bayes['income_needed'].quantile(0.75)
yearly = (
    bayes.assign(high_stress=(bayes['income_needed'] > threshold).astype(int), year=bayes['date'].dt.year)
    .groupby('year', as_index=False)
    .agg(
        income_needed=('income_needed', 'mean'),
        mortgage=('MORTGAGE30US', 'mean'),
        inventory=('inventory', 'mean'),
        days_to_pending=('days_to_pending', 'mean'),
        high_stress_rate=('high_stress', 'mean'),
    )
)

print(f'75th percentile income_needed threshold: ${threshold:,.0f}')
print('\nYearly feature summary')
print(yearly.round(2).to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)

axes[0, 0].plot(monthly['month'], monthly['income_needed'], color='steelblue', linewidth=2.5)
axes[0, 0].axhline(threshold, color='gray', linestyle='--', linewidth=2)
axes[0, 0].set_title('Income needed')
axes[0, 0].set_ylabel('Dollars')
axes[0, 0].grid(True, alpha=0.25)

axes[0, 1].plot(monthly['month'], monthly['mortgage'], color='darkorange', linewidth=2.5)
axes[0, 1].set_title('30-year mortgage rate')
axes[0, 1].set_ylabel('Percent')
axes[0, 1].grid(True, alpha=0.25)

axes[1, 0].plot(monthly['month'], monthly['inventory'], color='seagreen', linewidth=2.5)
axes[1, 0].set_title('Inventory')
axes[1, 0].set_ylabel('Listings')
axes[1, 0].grid(True, alpha=0.25)

axes[1, 1].plot(monthly['month'], monthly['days_to_pending'], color='firebrick', linewidth=2.5)
axes[1, 1].set_title('Days to pending')
axes[1, 1].set_ylabel('Days')
axes[1, 1].grid(True, alpha=0.25)

for ax in axes.flat:
    ax.set_xlabel('Date')

plt.suptitle('The affordability features move together into a new regime after 2021', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()